# Governance Template: Dataset Card, Model Card, Reproducibility Checklist

## Notebook Contract
- **Objective:** demonstrate the v0.9 governance workflow — produce a dataset card, a task-specific model card with curated limitations, and a reproducibility checklist from one short training run.
- **Inputs:** synthetic support-ticket data generated locally; a TF-IDF + LogReg baseline trained inside the notebook.
- **Outputs:** `dataset_card.md`, `model_card.md`, `reproducibility.md`, and `reproducibility.json` written under `reports/sample_run/governance/`.
- **Expected runtime:** under 30 seconds on CPU.
- **Scope boundary:** templates and writers live in `src/hf_finetuning_lab/governance/`; this notebook orchestrates and previews the rendered artifacts.

## 1) Setup and Reproducibility

In [1]:
from __future__ import annotations

import os
import platform
import random
import sys
from datetime import UTC, datetime
from pathlib import Path

ROOT = Path('..').resolve()
SRC_PATH = ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from hf_finetuning_lab.data.io import validate_text_classification_frame
from hf_finetuning_lab.experiments import hash_dataframe, make_run_id
from hf_finetuning_lab.governance import (
    DatasetCard,
    DatasetColumn,
    DatasetSplit,
    ReproducibilityRecord,
    capture_environment,
    task_limitations,
    write_dataset_card,
    write_reproducibility_checklist,
    write_task_model_card,
)
from hf_finetuning_lab.sample_data import write_sample_data

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

print(f'Python: {sys.version.split()[0]}')
print(f'Platform: {platform.platform()}')
print(f"Timestamp (UTC): {datetime.now(UTC).isoformat(timespec='seconds')}")
print(f'Seed: {SEED}')

Python: 3.12.11
Platform: Windows-11-10.0.26200-SP0
Timestamp (UTC): 2026-05-19T11:33:19+00:00
Seed: 42


## 2) Parameters

In [2]:
DATA_PATH = ROOT / 'data' / 'raw' / 'support_tickets.csv'
GOVERNANCE_DIR = ROOT / 'reports' / 'sample_run' / 'governance'
DATASET_CARD_PATH = GOVERNANCE_DIR / 'dataset_card.md'
MODEL_CARD_PATH = GOVERNANCE_DIR / 'model_card.md'
REPRO_PATH = GOVERNANCE_DIR / 'reproducibility.md'

TEXT_COL = 'text'
LABEL_COL = 'label'
ROWS = 800
TEST_SIZE = 0.2
GOVERNANCE_DIR.mkdir(parents=True, exist_ok=True)

print(f'Governance dir: {GOVERNANCE_DIR}')

Governance dir: C:\Users\diogo\work_code\portfolio\huggingface-finetuning-lab\reports\sample_run\governance


## 3) Train a Small Model on Synthetic Data

We reuse the synthetic support-ticket dataset and train one TF-IDF + LogReg baseline so the model card and reproducibility checklist have real metrics, a real dataset hash, and a real run ID to point at.

In [3]:
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
write_sample_data(output=DATA_PATH, rows=ROWS, seed=SEED)
df = pd.read_csv(DATA_PATH)
validate_text_classification_frame(df, text_col=TEXT_COL, label_col=LABEL_COL)
DATASET_HASH = hash_dataframe(df, columns=[TEXT_COL, LABEL_COL])

train_df, test_df = train_test_split(df, test_size=TEST_SIZE, random_state=SEED, stratify=df[LABEL_COL])

pipeline = Pipeline(
    steps=[
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=12000)),
        ('clf', LogisticRegression(max_iter=1000, random_state=SEED)),
    ]
)
pipeline.fit(train_df[TEXT_COL], train_df[LABEL_COL])
pred = pipeline.predict(test_df[TEXT_COL])
metrics = {
    'macro_f1': float(f1_score(test_df[LABEL_COL], pred, average='macro', zero_division=0)),
    'weighted_f1': float(f1_score(test_df[LABEL_COL], pred, average='weighted', zero_division=0)),
}
RUN_ID = make_run_id(prefix='governance-demo')
print(f'Run ID:        {RUN_ID}')
print(f'Dataset hash:  {DATASET_HASH[:16]}…')
print(f'Macro F1:      {metrics["macro_f1"]:.4f}')
print(f'Weighted F1:   {metrics["weighted_f1"]:.4f}')

Run ID:        governance-demo-20260519T113319Z-6d26b5
Dataset hash:  fafc0c83a222c6c5…
Macro F1:      1.0000
Weighted F1:   1.0000


## 4) Dataset Card

Capture schema, splits, label distribution, intended use, out-of-scope use, privacy notes, and limitations in a structured way. Per-split label counts come straight from the train/test frames.

In [4]:
def _label_counts(frame: pd.DataFrame) -> dict[str, int]:
    return {str(k): int(v) for k, v in frame[LABEL_COL].value_counts().items()}


dataset_card = DatasetCard(
    name='synthetic-support-triage',
    description=(
        'Synthetic customer-support ticket triage data generated by '
        '`hf_finetuning_lab.sample_data`. The text and labels are illustrative; '
        'they do not reflect any real operational taxonomy.'
    ),
    task='text-classification',
    columns=[
        DatasetColumn(name='id', dtype='int', role='identifier', description='Row index assigned at generation time.'),
        DatasetColumn(name=TEXT_COL, dtype='str', role='feature', description='Free-text ticket body.'),
        DatasetColumn(name=LABEL_COL, dtype='str', role='target', description='Triage category.'),
    ],
    splits=[
        DatasetSplit(name='train', n_rows=len(train_df), label_distribution=_label_counts(train_df)),
        DatasetSplit(name='test', n_rows=len(test_df), label_distribution=_label_counts(test_df)),
    ],
    collection_method='synthetic (rule-based templates plus light noise)',
    license='internal',
    privacy_notes=[
        'No real customer data; every example is generated from templates.',
        'No personally identifying information is embedded in the text.',
    ],
    intended_use=[
        'Workflow demonstrations across the rest of the lab notebooks.',
        'CI smoke runs that need a stable, small text-classification dataset.',
    ],
    not_intended_use=[
        'Production routing or prioritisation of real support tickets.',
        'Benchmarking model quality — the synthetic labels are illustrative only.',
    ],
    limitations=[
        'Labels are synthetic and do not reflect any real operational taxonomy.',
        'Class balance is fixed; real triage distributions drift over time.',
        'Text vocabulary is narrow because it is generated from templates.',
    ],
    sources=['src/hf_finetuning_lab/sample_data.py'],
)
write_dataset_card(dataset_card, DATASET_CARD_PATH)
print(f'Wrote dataset card to: {DATASET_CARD_PATH}')
print(DATASET_CARD_PATH.read_text(encoding='utf-8')[:900])

Wrote dataset card to: C:\Users\diogo\work_code\portfolio\huggingface-finetuning-lab\reports\sample_run\governance\dataset_card.md
# Dataset Card: synthetic-support-triage

## Overview

- **Task:** text-classification
- **Collection method:** synthetic (rule-based templates plus light noise)
- **License:** internal
- **Created:** 2026-05-19

## Description

Synthetic customer-support ticket triage data generated by `hf_finetuning_lab.sample_data`. The text and labels are illustrative; they do not reflect any real operational taxonomy.

## Schema

| name | dtype | role | description |
| --- | --- | --- | --- |
| `id` | `int` | identifier | Row index assigned at generation time. |
| `text` | `str` | feature | Free-text ticket body. |
| `label` | `str` | target | Triage category. |

## Splits

### train

- Rows: **640**
- Label distribution:
  - `account`: 122
  - `billing`: 109
  - `cancellation`: 78
  - `delivery`: 75
  - `security`: 124
  - `technical`: 132

### test

- Rows: **160**
-

## 5) Task-Specific Model Card

`write_task_model_card` injects the curated v0.4 / v0.7 / v0.8 limitations for the chosen task. Project-specific caveats are appended through `extra_limitations`.

In [5]:
print('Curated limitations for `text-classification`:')
for bullet in task_limitations('text-classification'):
    print(f'  - {bullet}')

write_task_model_card(
    output_path=MODEL_CARD_PATH,
    model_name='tfidf-logreg-governance-demo',
    task='text-classification',
    label_names=sorted(test_df[LABEL_COL].unique().tolist()),
    metrics=metrics,
    extra_limitations=[
        'Dataset is synthetic; numbers are not a proxy for production performance.',
        f'Trained on a single seed ({SEED}); pair with the v0.4 bootstrap CIs before relying on point estimates.',
    ],
)
print(f'\nWrote model card to: {MODEL_CARD_PATH}')
print(MODEL_CARD_PATH.read_text(encoding='utf-8')[:900])

Curated limitations for `text-classification`:
  - Aggregate metrics can hide poor subgroup performance; pair this card with the v0.4 robust-evaluation report.
  - Predicted-label distribution may drift when input distributions shift; track PSI between deployment windows.
  - Default decision thresholds rarely maximise F1 — tune per class on a held-out validation set.

Wrote model card to: C:\Users\diogo\work_code\portfolio\huggingface-finetuning-lab\reports\sample_run\governance\model_card.md
# Model Card: tfidf-logreg-governance-demo

## Overview

- **Base model:** `tfidf-logreg-governance-demo`
- **Task:** text-classification
- **Created:** 2026-05-19

## Labels

- account
- billing
- cancellation
- delivery
- security
- technical

## Evaluation Metrics

- **macro_f1**: 1.0000
- **weighted_f1**: 1.0000

## Intended Use

This model is intended for text-classification experiments and workflow demonstrations. Real deployment requires validation on the target domain.

## Limitations

- 

## 6) Reproducibility Checklist

Captures environment (Python, platform, git commit + dirty flag), seed, dataset hash, training config, metrics, and a tick-box checklist. A JSON sidecar with the raw record is written next to the Markdown checklist for machine consumption.

In [6]:
environment = capture_environment(cwd=ROOT)
training_config = {
    'vectorizer.ngram_range': '(1, 2)',
    'vectorizer.max_features': 12000,
    'classifier.max_iter': 1000,
    'classifier.random_state': SEED,
    'test_size': TEST_SIZE,
    'rows': ROWS,
}
record = ReproducibilityRecord(
    run_id=RUN_ID,
    task='text-classification',
    seed=SEED,
    dataset_hash=DATASET_HASH,
    model_name='tfidf-logreg-governance-demo',
    config=training_config,
    metrics=metrics,
    environment=environment,
    notes='Generated by notebooks/07_governance_template.ipynb.',
)
write_reproducibility_checklist(record, REPRO_PATH)
print(f'Wrote reproducibility checklist to: {REPRO_PATH}')
print(f'JSON sidecar:                       {REPRO_PATH.with_suffix(".json")}')
print()
print(REPRO_PATH.read_text(encoding='utf-8'))

Wrote reproducibility checklist to: C:\Users\diogo\work_code\portfolio\huggingface-finetuning-lab\reports\sample_run\governance\reproducibility.md
JSON sidecar:                       C:\Users\diogo\work_code\portfolio\huggingface-finetuning-lab\reports\sample_run\governance\reproducibility.json

# Reproducibility Checklist: governance-demo-20260519T113319Z-6d26b5

## Overview

- **Run ID:** `governance-demo-20260519T113319Z-6d26b5`
- **Task:** text-classification
- **Model:** `tfidf-logreg-governance-demo`
- **Seed:** `42`
- **Dataset hash:** `fafc0c83a222c6c55a72698ef5497040c02b2ddd347e363c82cf7fbd3956f781`

## Environment

- **Python:** `3.12.11`
- **Platform:** `Windows-11-10.0.26200-SP0`
- **Captured at (UTC):** `2026-05-19T11:33:19+00:00`
- **Git commit:** `3116e0c`
- **Git dirty:** `True`

## Configuration

- `classifier.max_iter`: `1000`
- `classifier.random_state`: `42`
- `rows`: `800`
- `test_size`: `0.2`
- `vectorizer.max_features`: `12000`
- `vectorizer.ngram_range`: `(1, 2)

## 7) Checklist
- Three artifacts written under `reports/sample_run/governance/`: dataset card, model card, reproducibility checklist (plus JSON sidecar).
- The model card uses curated task-specific limitations from `governance.templates`; switch the `task` argument to `token-classification` or `retrieval` to get the v0.7 / v0.8 limitations.
- The reproducibility record ties together: run ID (`experiments.make_run_id`), dataset hash (`experiments.hash_dataframe`), environment snapshot, training config, and metrics. Re-running the notebook with the same seed should regenerate the same dataset hash and metrics.
- For production: extend the dataset card with provenance + retention notes, add subgroup metrics from the v0.4 notebook to the model card, and gate model promotion on a clean reproducibility checklist.